In [ ]:
import tifffile as tiff
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def print_row_info(row):
    print("Processing data_id: " + str(row.data_id))
    print("Construction cost per m2 usd: " + str(row.construction_cost_per_m2_usd))
    print("Year: " + str(row.year) + " | Quarter: " + str(row.quarter_label))
    print("Country: " + str(row.country) + " | geolocation: " + str(row.geolocation_name))
    print("Economic data:")
    print("  deflated_gdp_usd: " + str(row.deflated_gdp_usd))
    print("  us_cpi: " + str(row.us_cpi))
    print("  developed_country: " + str(row.developed_country))
    print("  economic_classification: " + str(row.region_economic_classification))
    print("Infrastructure access:")
    print("  landlocked: " + str(row.landlocked))
    print("  access_to_airport: " + str(row.access_to_airport))
    print("  access_to_port: " + str(row.access_to_port))
    print("  access_to_highway: " + str(row.access_to_highway))
    print("  access_to_railway: " + str(row.access_to_railway))
    print("Geographical data:")
    print("  straight_distance_to_capital_km: " + str(row.straight_distance_to_capital_km))
    print("  seismic_hazard_zone: " + str(row.seismic_hazard_zone))
    print("  flood_risk_class: " + str(row.flood_risk_class))
    print("  tropical_cyclone_wind_risk: " + str(row.tropical_cyclone_wind_risk))
    print("  tornadoes_wind_risk: " + str(row.tornadoes_wind_risk))
    print("  koppen_climate_zone: " + str(row.koppen_climate_zone))

def display_img(img, title=None, cmap=None):
    plt.figure(figsize=(4,4))
    plt.imshow(img, cmap=cmap)
    if title:
        plt.title(title)
    plt.axis('off')
    plt.show()

#display images in an n x m grid
def display_multi_img(images, n, m):
    fig, axs = plt.subplots(n, m, figsize=(15, 10))
    for i in range(n):
        for j in range(m):
            idx = i * m + j
            if idx < len(images):
                img, title, cmap = images[idx]
                axs[i, j].imshow(img, cmap=cmap)
                if title:
                    axs[i, j].set_title(title)
                axs[i, j].axis('off')
            else:
                axs[i, j].axis('off')

    plt.tight_layout()
    plt.show()

def convert_to_rgb(band):
    band_min = np.nanmin(band)
    band_max = np.nanmax(band)
    band_normalized = (band - band_min) / (band_max - band_min)
    return band_normalized

def convert_rgb_img(bands):
    rgb_bands = []
    for band in bands:
        rgb_band = convert_to_rgb(band)
        rgb_bands.append(rgb_band)
    rgb_img = np.stack([rgb_bands[0], rgb_bands[1], rgb_bands[2]], axis=-1)
    return rgb_img

DataPath = Path("..") / "Training data"

train_tabular = pd.read_csv(DataPath / "train_tabular.csv")
#print(train_tabular.head())
print(f"Tabular shape: {train_tabular.shape}")

print(train_tabular.columns)
print()

for row in train_tabular.head().itertuples(index=False):
    print_row_info(row)

    print("Image files:")
    imgPath = DataPath / "train_composite"
    images = []
    viirs = tiff.imread(imgPath / row.viirs_tiff_file_name)
    print("Viirs: " + str(viirs.shape))
    images.append((viirs, "VIIRS", None))

    sentinel = tiff.imread(imgPath / row.sentinel2_tiff_file_name)
    #RGB from Sentinel bands B4, B3, B2
    rgb_img = convert_rgb_img([sentinel[:, :, 3], sentinel[:, :, 2], sentinel[:, :, 1]])
    images.append((rgb_img, "RGB", None))
    #Color Infrared from Sentinel bands B8, B4, B3
    rgb_img = convert_rgb_img([sentinel[:, :, 7], sentinel[:, :, 3], sentinel[:, :, 2]])
    images.append((rgb_img, "Color Infrared", None))
    #Shortwave Infrared from Sentinel bands B12, B8A, B4
    rgb_img = convert_rgb_img([sentinel[:, :, 11], sentinel[:, :, 8], sentinel[:, :, 3]])
    images.append((rgb_img, "Shortwave Infrared", None))
    #display_img(rgb_img, title="Shortwave Infrared")
    #Agriculture from Sentinel bands B11, B8, B2
    rgb_img = convert_rgb_img([sentinel[:, :, 10], sentinel[:, :, 7], sentinel[:, :, 1]])
    images.append((rgb_img, "Agriculture", None))
    #display_img(rgb_img, title="Agriculture")
    #Geology from Sentinel bands B12, B11, B2
    rgb_img = convert_rgb_img([sentinel[:, :, 11], sentinel[:, :, 10], sentinel[:, :, 1]])
    images.append((rgb_img, "Geology", None))
    #display_img(rgb_img, title="Geology", )
    #Vegetation Index from Sentinel bands (B8 - B4) / (B8 + B4)
    vegetation_index = (sentinel[:, :, 7] - sentinel[:, :, 3]) / (sentinel[:, :, 7] + sentinel[:, :, 3])
    images.append((vegetation_index, "Vegetation Index", 'RdYlGn'))
    #display_img(vegetation_index, title="Vegetation Index", cmap='RdYlGn')
    #Moisture Index from Sentinel bands (B8A - B11) / (B8A + B11)
    moisture_index = (sentinel[:, :, 8] - sentinel[:, :, 11]) / (sentinel[:, :, 8] + sentinel[:, :, 11])
    images.append((moisture_index, "Moisture Index", 'RdYlBu'))
    #display_img(moisture_index, title="Moisture Index", cmap='RdYlBu')

    print("Sentinel: " + str(sentinel.shape))
    display_multi_img(images, 2, 4)
    sentinel_img = [(sentinel[:,:,x], f"Band {x + 1}", None) for x in range(sentinel.shape[2])]
    display_multi_img(sentinel_img, 3, 4)

In [ ]:
japan = train_tabular[train_tabular['country'] == 'Japan']
philipines = train_tabular[train_tabular['country'] == 'Philippines']

datasets = [("Japan", japan), ("Philippines", philipines)]

# Get numeric columns only
numeric_columns = japan.select_dtypes(include=[np.number]).columns.tolist()
numeric_columns.remove('year')


# Create a separate plot for each numeric column
for col in numeric_columns:
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    
    for idx, (key, dataset) in enumerate(datasets):
        axs[idx].hist(dataset[col].dropna(), bins=30, color='blue', alpha=0.7)
        axs[idx].set_title(f'{key} - {col}')
        axs[idx].set_xlabel(col)
        axs[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

# Get non numeric columns only
non_numeric_columns = japan.select_dtypes(exclude=[np.number]).columns.tolist()

non_numeric_columns.remove('viirs_tiff_file_name')
non_numeric_columns.remove('sentinel2_tiff_file_name')
non_numeric_columns.remove('data_id')
non_numeric_columns.remove('country')

# Create a separate plot for each non-numeric column
for col in non_numeric_columns:
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))
    
    for idx, (key, dataset) in enumerate(datasets):
        dataset[col].value_counts().plot(kind='bar', ax=axs[idx], color='blue', alpha=0.7)
        axs[idx].set_title(f'{key} - {col}')
        axs[idx].set_xlabel(col)
        axs[idx].set_ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()